In [ ]:
!wget 'https://os.unil.cloud.switch.ch/fma/fma_small.zip'

In [ ]:
!mkdir '/kaggle/working/fma_small'

In [ ]:
%%writefile input
A

In [ ]:
!7za x '/kaggle/working/fma_small.zip' < input

In [ ]:
!sudo apt-get install megatools

In [ ]:
%matplotlib inline

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("PYTORCH_CUDA_ALLOC_CONF =", os.environ.get("PYTORCH_CUDA_ALLOC_CONF"))
import random
import IPython.display as ipd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn as skl
import sklearn.utils, sklearn.preprocessing, sklearn.decomposition, sklearn.svm
import librosa
import librosa.display
import IPython
import pickle
from scipy import signal
from scipy.fft import fft, fftshift
plt.rcParams['figure.figsize'] = (17, 5)

-----------------

TRAINLOADER


In [ ]:
from pathlib import Path
TMP_DIR = Path('../temp')
TMP_DIR.mkdir(exist_ok=True)

In [ ]:
!mkdir /kaggle/temp/train_v2

In [ ]:
!mkdir /kaggle/temp/test_v2

In [ ]:
!mkdir /kaggle/temp/validation_v2

In [ ]:
ls /kaggle/temp/test_v2

In [ ]:
def save_on_drive(data_list, dataset, dir_name):

    if dataset == 'train':
        path = '/kaggle/temp/train_v2'
    elif dataset == 'test':
        path = '/kaggle/temp/test_v2'
    elif dataset == 'validation':
        path = '/kaggle/temp/validation_v2'

    print("SAVING.....")
    for i, data in enumerate(data_list):
        obj = str(i) + "_" + dir_name + ".mus"
        filename = os.path.join(path, obj)
        with open(filename, 'wb') as fd:
            pickle.dump(data, fd)

In [ ]:
import os

file_path = '/kaggle/working/fma_small.zip'

if os.path.exists(file_path):
    os.remove(file_path)
    print(f"File eliminato: {file_path}")
else:
    print(f"Il file non esiste: {file_path}")

In [ ]:
import warnings
warnings.filterwarnings("ignore")

path_dir = '/kaggle/working/fma_small/'

COLS=1024

num_cart = 30
names = set()
target_length = 22050 * 30

for dir in os.listdir(path_dir):
    if num_cart <= 0:
        break
    else:
        num_cart = num_cart -1
        names.add(dir)

    if os.path.isdir(os.path.join(path_dir, dir)):

        data_list = []

        subpath = os.path.join(path_dir, dir)
        print(subpath)
        for i, song in enumerate(os.listdir(subpath)):
            try:
                data, samplerate = librosa.load(os.path.join(subpath, song), sr=None)
                data = librosa.util.fix_length(data, size=target_length)
                data_list.append(data)
            except Exception as e:
                print(f'Errore in directory {dir}, file: {song}. Error: {e}')


        tot_songs = len(data_list)
        if tot_songs < 10:
            continue

        random_indexes = list(range(0, tot_songs))
        random.shuffle(random_indexes)

        train_len = round(tot_songs*0.7)
        test_len = round(tot_songs*0.15)

        data_array = np.asarray(data_list)


        train = data_array[random_indexes[:train_len]]
        test = data_array[random_indexes[train_len:test_len+train_len]]
        validation = data_array[random_indexes[test_len+train_len:]]

        save_on_drive(train, 'train', str(dir))
        save_on_drive(test, 'test', str(dir))
        save_on_drive(validation, 'validation', str(dir))

In [ ]:
!ls /kaggle/temp/train_v2 | wc -l


In [ ]:
import torch

class Load_Dataset(torch.utils.data.Dataset):

    def __init__(self, path, ext):
        self.path = path
        self.ext = ext
        self.len = len(os.listdir(self.path))


    def __len__(self):
        return self.len


    def __getitem__(self, idx):
        name = self.path + '/' + str(idx) + "." + self.ext

        with open(name, 'rb') as fd:
            image = pickle.load(fd)

        return image

In [ ]:
!du -h /content/train_v2

In [ ]:
import os
import pickle
import numpy as np


train_dir = '/kaggle/temp/train_v2'

wave_dir = '/kaggle/temp/wave'


if not os.path.exists(wave_dir):
    os.makedirs(wave_dir)
    print(f"Creata directory: {wave_dir}")


for filename in os.listdir(train_dir):
    if os.path.isfile(os.path.join(train_dir, filename)) and filename.endswith('.mus'):
        filepath = os.path.join(train_dir, filename)
        try:
            with open(filepath, 'rb') as f:
                loaded_object = pickle.load(f)
                print("Type:", type(loaded_object))
            if isinstance(loaded_object, tuple):
                print("Lunghezza tupla:", len(loaded_object))
                for i, el in enumerate(loaded_object[:3]):
                    print(f" Elemeto {i} type:", type(el), "shape:", getattr(el, "shape", None))
            elif isinstance(loaded_object, np.ndarray):
                print(" ndarray shape:", loaded_object.shape, "dtype:", loaded_object.dtype)
            else:
                print(" other object; repr:", repr(loaded_object)[:200])

            if isinstance(loaded_object, tuple) and len(loaded_object) >= 1:
                y = loaded_object[0]
            elif isinstance(loaded_object, np.ndarray):
                y = loaded_object
            else:
                print(f"Salto {filename}: Unexpected object type {type(loaded_object)}")
                continue


            npy_filename = os.path.splitext(filename)[0] + '.npy'
            npy_filepath = os.path.join(wave_dir, npy_filename)


            try:
                np.save(npy_filepath, y)
                print(f"Salvato {filename} come {npy_filename}")
            except Exception as e:
                print(f"Errore in {npy_filepath}: {e}")

        except Exception as e:
            print(f"Errore {filename}: {e}")

In [ ]:
ls /kaggle/temp/wave

In [ ]:
import os
import pickle
import numpy as np
import librosa

train_dir = '/kaggle/temp/train_v2'
stft_dir = '/kaggle/temp/stft'


if not os.path.exists(stft_dir):
    os.makedirs(stft_dir)
    print(f"Creata directory: {stft_dir}")

for filename in os.listdir(train_dir):
    if os.path.isfile(os.path.join(train_dir, filename)) and filename.endswith('.mus'):
        filepath = os.path.join(train_dir, filename)
        try:
            with open(filepath, 'rb') as f:
                loaded_object = pickle.load(f)

            if isinstance(loaded_object, tuple) and len(loaded_object) >= 1:
                y = loaded_object[0]
                sr = loaded_object[1] if len(loaded_object) > 1 else 22050
            elif isinstance(loaded_object, np.ndarray):
                y = loaded_object
                sr = 22050
            else:
                print(f"Salto {filename}: Unexpected object type {type(loaded_object)}")
                continue

            try:
                y = y.astype(np.float32)
                n_fft = 4096
                win_length = 4096
                stft = librosa.stft(y, n_fft=n_fft, win_length=win_length, window=signal.windows.hamming(win_length))

                stft_filename = os.path.splitext(filename)[0] + '.npy'
                stft_filepath = os.path.join(stft_dir, stft_filename)

                np.save(stft_filepath, stft)
                print(f"Calcolata e salvata STFT per {filename} come {stft_filename} con shape {stft.shape}")

            except Exception as e:
                print(f"Errore nel calcolo della STFT {filename}: {e}")

        except Exception as e:
            print(f"Errore {filename}: {e}")

# Codice altro tesista


In [ ]:
from tqdm import tqdm
import pickle
import os
import librosa
import librosa.display
import IPython.display as ipd
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.signal import butter, lfilter, freqz

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Parametri globali
sample_rate = 22050  # sample rate degli audio
cutoff = 7000  # frequenza di cutoff
max_len_wave = 661560
stride = 1024
window_size = 4096

def butter_lowpass(cutoff, fs, order=2):
    return butter(order, cutoff, fs=fs, btype='low', analog=False)

def butter_lowpass_filter(data, cutoff, fs, order=2):
    b, a = butter_lowpass(cutoff, fs, order=order)
    y = lfilter(b, a, data)
    #print(y.size())
    return y

def remove_window(audio_windowed):
    n_frames = audio_windowed.shape[0]
    output_len = (n_frames - 1) * stride + window_size

    out = np.zeros(661500)

    out[0:window_size] = audio_windowed[0]

    for i in range(1, n_frames):
        start = i * stride
        end = min(start + window_size, 661500)
        window_len = end - start

        if window_len > 0:
            overlap_start = max(0, start)
            overlap_end = min(start + stride, 661500)

            if overlap_end > overlap_start:
                out[overlap_start:overlap_end] = (out[overlap_start:overlap_end] +
                                                   audio_windowed[i][:overlap_end-overlap_start]) / 2
            if overlap_end < end:
                out[overlap_end:end] = audio_windowed[i][overlap_end-start:window_len]

    return out

def frame_audio(audio, frame_len=4096, hop=1024):
    n_frames = 1 + (len(audio) - frame_len) // hop
    frames = np.zeros((n_frames, frame_len), dtype=audio.dtype)
    for i in range(n_frames):
        start = i * hop
        end = start + frame_len
        frames[i] = audio[start:end]
    return frames


# ============== DATASET ==============

class AudioSTFTDataset(torch.utils.data.Dataset):
    def __init__(self, audio_paths, stft_paths):
        """
        Dataset che gestisce correttamente audio e STFT.

        Args:
            audio_paths (list): Lista dei path delle tracce audio (.npy files).
            stft_paths (list): Lista dei path delle STFT corrispondenti (.npy files).

        Returns:
            audio_filtered_window: Audio filtrato e diviso in finestre [n_frames, window_size]
            stft_filtered: STFT dell'audio filtrato
            stft_gt: STFT ground truth (originale)
            audio_gt: Audio ground truth (originale, non windowed)
        """
        self.audio_paths = audio_paths
        self.stft_paths = stft_paths

        self.valid_indices = []
        for i, (audio_path, stft_path) in enumerate(zip(audio_paths, stft_paths)):
            if os.path.exists(audio_path) and os.path.exists(stft_path):
                self.valid_indices.append(i)

        if len(self.valid_indices) == 0:
            raise ValueError("Nessun file valido trovato nei path specificati!")

        print(f"Dataset inizializzato con {len(self.valid_indices)} file validi su {len(audio_paths)} totali")

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        real_idx = self.valid_indices[idx]

        try:
            raw_audio = np.load(self.audio_paths[real_idx])
        except Exception as e:
            print(f"Errore nel caricare {self.audio_paths[real_idx]}: {e}")
            return (torch.zeros(1, window_size),
                    torch.zeros(2049, 1, dtype=torch.complex64),
                    torch.zeros(2049, 1, dtype=torch.complex64),
                    np.zeros(661500))

        if len(raw_audio) > 661500:
            raw_audio = raw_audio[:661500]
        elif len(raw_audio) < 661500:
            raw_audio = np.pad(raw_audio, (0, 661500 - len(raw_audio)), mode='constant')

        filtered_audio = butter_lowpass_filter(raw_audio, cutoff, sample_rate, order=2)

        audio_windowed = frame_audio(filtered_audio, frame_len=window_size, hop=stride)

        try:
            stft_gt = np.load(self.stft_paths[real_idx])
        except Exception as e:
            print(f"Errore nel caricare STFT {self.stft_paths[real_idx]}: {e}")
            stft_gt = librosa.stft(
                raw_audio,
                n_fft=4096,
                win_length=4096,
                window='hann'
            )

        stft_train = librosa.stft(
            filtered_audio,
            n_fft=4096,
            win_length=4096,
            window='hann'
        )

        audio_tensor = torch.tensor(audio_windowed.astype(np.float32))
        stft_gt_tensor = torch.tensor(stft_gt.astype(np.complex64))
        stft_train_tensor = torch.tensor(stft_train.astype(np.complex64))

        return audio_tensor, stft_train_tensor, stft_gt_tensor, raw_audio


In [ ]:
# --------------- MODELLO ---------------
class ReteRecWithAttention(nn.Module):
    def __init__(self, window_size, freq_bins, hidden_size=128, dropout_p=0.2):
        super(ReteRecWithAttention, self).__init__()
        self.window_size = window_size
        self.freq_bins = freq_bins

        self.rec_imag_1 = nn.LSTM(self.freq_bins, hidden_size*2, num_layers=1, batch_first=True)
        self.rec_real_1 = nn.LSTM(self.freq_bins, hidden_size*2, num_layers=1, batch_first=True)

        self.rec_imag_2 = nn.LSTM(hidden_size*2, hidden_size, num_layers=1, batch_first=True)
        self.rec_real_2 = nn.LSTM(hidden_size*2, hidden_size, num_layers=1, batch_first=True)

        self.rec_dec_1 = nn.LSTM(hidden_size*2, hidden_size*2, num_layers=1, batch_first=True)
        self.rec_dec_2 = nn.LSTM(hidden_size*2, hidden_size*2, num_layers=1, batch_first=True)

        self.proiezione_imag_1 = nn.Linear(hidden_size*2, self.freq_bins//2)
        self.proiezione_imag_2 = nn.Linear(self.freq_bins//2, self.freq_bins)
        self.proiezione_real_1 = nn.Linear(hidden_size*2, self.freq_bins//2)
        self.proiezione_real_2 = nn.Linear(self.freq_bins//2, self.freq_bins)

        self.init_layers()

    def init_layers(self):
        for name, param in self.named_parameters():
            if 'weight' in name:
                if 'rec_' in name and 'weight_hh' in name:
                    nn.init.orthogonal_(param.data)
                elif 'rec_' in name and 'weight_ih' in name:
                    nn.init.orthogonal_(param.data)
                elif 'proiezione' in name:
                    nn.init.xavier_normal_(param.data)
            elif 'bias' in name:
                param.data.fill_(0.0)

    def forward(self, real, imag):
        real = real.to(device)
        imag = imag.to(device)

        out_imag = F.leaky_relu(self.rec_imag_2(F.leaky_relu(self.rec_imag_1(imag)[0]))[0])
        out_real = F.leaky_relu(self.rec_real_2(F.leaky_relu(self.rec_real_1(real)[0]))[0])

        encoder_output = torch.cat([out_imag, out_real], dim=2)

        outputs_real = torch.zeros(real.size(1), real.size(0), real.size(2)).to(device)
        outputs_imag = torch.zeros(imag.size(1), imag.size(0), imag.size(2)).to(device)

        decoder_input = encoder_output[:, -1, :].squeeze(1) if encoder_output.dim() > 2 else encoder_output[:, -1, :]

        for t in range(real.size(1)):
            if decoder_input.dim() == 1:
                decoder_input = decoder_input.unsqueeze(0)
            if decoder_input.dim() == 2:
                decoder_input = decoder_input.unsqueeze(1)

            decoder_output_1 = F.leaky_relu(self.rec_dec_1(decoder_input)[0])
            decoder_output = F.leaky_relu(self.rec_dec_2(decoder_output_1)[0])

            if decoder_output.dim() == 3:
                decoder_output = decoder_output.squeeze(1)
            decoder_output = decoder_output + encoder_output[:, t, :]

            out_proj_real = F.leaky_relu(self.proiezione_real_2(F.leaky_relu(self.proiezione_real_1(decoder_output))))
            out_proj_imag = F.leaky_relu(self.proiezione_imag_2(F.leaky_relu(self.proiezione_imag_1(decoder_output))))

            outputs_real[t] = out_proj_real
            outputs_imag[t] = out_proj_imag
            decoder_input = decoder_output.unsqueeze(1)

        outputs_real = outputs_real.permute(1, 0, 2)
        outputs_imag = outputs_imag.permute(1, 0, 2)

        return outputs_real, outputs_imag



In [ ]:
wave_dir = '/kaggle/temp/wave'
stft_dir='/kaggle/temp/stft'
audio_files = os.listdir(wave_dir)
stft_files = os.listdir(stft_dir)

audio_paths = [os.path.join(wave_dir, file) for file in audio_files]
stft_paths = [os.path.join(stft_dir, file) for file in stft_files]
dataset_prova = AudioSTFTDataset(audio_paths,stft_paths)
from torch.utils.data import random_split, DataLoader

total_size = len(dataset_prova)
train_size = int(0.8 * total_size)
val_size = int(0.1 * total_size)
test_size = total_size - train_size - val_size
gen = torch.Generator(device='cpu')
gen.manual_seed(13)
train_dataset, val_dataset, test_dataset = random_split(dataset_prova, [train_size, val_size, test_size],gen)


train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=20, shuffle=True,generator=gen,num_workers=0)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=20, shuffle=True,generator=gen,num_workers=0)
val_loader = torch.utils.data.DataLoader(dataset=val_dataset, batch_size=20, shuffle=True,generator=gen,num_workers=0)

In [ ]:
rete2=ReteRecWithAttention(window_size=4096,freq_bins=2049,hidden_size=128).to(device)

In [ ]:
import torch
import torch.nn.functional as F

class SpectralCoherenceLoss(nn.Module):
    def __init__(self):
        super(SpectralCoherenceLoss, self).__init__()
    def forward(self, real_pred, imag_pred, real_target, imag_target):
        stft_pred = torch.complex(real_pred, imag_pred)
        stft_target = torch.complex(real_target, imag_target)

        pred_diff = stft_pred[:, :, 1:] - stft_pred[:, :, :-1]
        #print("Pred_diff:",pred_diff.mean().detach())
        target_diff = stft_target[:, :, 1:] - stft_target[:, :, :-1]
        #print("Target_diff:",target_diff.mean().detach())


        #pred_power = pred_diff.real**2 + pred_diff.imag**2
        #print("Pred power:",pred_power)
        #target_power = target_diff.real**2 + target_diff.imag**2
        #print("Target power:",target_power)
        coherence_loss = F.mse_loss(pred_diff.abs(), target_diff.abs())
        #print("Coherence loss:",coherence_loss)

        return coherence_loss

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CombinedLoss(nn.Module):
    def __init__(self, alpha=1.0,beta=0.1):

        super(CombinedLoss, self).__init__()
        self.alpha = alpha
        self.beta = beta
        self.spectral_coherence_loss = SpectralCoherenceLoss()

    def forward(self, real_pred, imag_pred, real_target, imag_target):
        """
        real_pred, imag_pred: STFT predetta, shape (batch, time_steps, freq_bins)
        real_target, imag_target: STFT target, shape (batch, time_steps, freq_bins)
        """

        #print("Real pred:",real_pred)
        #print("Imag pred:",imag_pred)
        #print("Real target:",real_target)
        #print("Imag target:",imag_target)
        mse_loss = F.mse_loss(real_pred, real_target)+F.mse_loss(imag_pred, imag_target)

        # Calcolo della loss di coerenza spettrale
        coherence_loss = self.spectral_coherence_loss(real_pred, imag_pred, real_target, imag_target)
        #print("Coherence_loss:",coherence_loss.item())
        #print("MSE_loss:",mse_loss.item())
        # Loss combinata
        total_loss = self.alpha*mse_loss + self.beta * coherence_loss
        print(total_loss)

        return total_loss

In [ ]:
loss=CombinedLoss(alpha=1.0,beta=0.3)
print(device)

In [ ]:
from tqdm import tqdm

def train(train_loader, val_loader, optimizer, loss, epoches, rete, device=device):
    num_batches = len(train_loader)
    losses_train = []
    losses_val = []
    losses_in_batches = []

    for epoca in range(epoches):
        running_loss = 0.0
        rete.train()

        for databatch in tqdm(train_loader):
            x_train = databatch[1].permute(0, 2, 1).to(device)
            y_train = databatch[2].permute(0, 2, 1).to(device)

            x_train_real = x_train.real
            x_train_imag = x_train.imag
            y_train_real = y_train.real
            y_train_imag = y_train.imag

            optimizer.zero_grad()
            output_real, output_imag = rete(x_train_real, x_train_imag)

            loss_value = loss(output_real, output_imag, y_train_real, y_train_imag)
            loss_value.backward()

            losses_in_batches.append(loss_value.item())
            running_loss += loss_value.item()
            optimizer.step()

        losses_train.append(running_loss/num_batches)
        val_loss = validate(val_loader, rete, loss)
        print(f"Train Loss all'epoca {epoca+1}: {running_loss/num_batches}")
        print(f"Validation Loss all'epoca {epoca+1}: {val_loss}")
        losses_val.append(val_loss)

    print("Training completato!")
    return losses_train, losses_val, losses_in_batches

def validate(val_loader, rete, loss):
    rete.eval()
    val_loss = 0.0
    num_batches = len(val_loader)

    with torch.no_grad():
        for databatch in tqdm(val_loader):
            x_val = databatch[1].permute(0, 2, 1).to(device)
            y_val = databatch[2].permute(0, 2, 1).to(device)

            output_real, output_imag = rete(x_val.real.to(device), x_val.imag.to(device))

            batch_loss = loss(output_real, output_imag, y_val.real.to(device), y_val.imag.to(device))
            val_loss += batch_loss.item()

    rete.train()
    return val_loss / num_batches


# Discriminatore

In [ ]:
class SubDiscriminator(nn.Module):
    """
    Sotto-discriminatore per una singola scala.
    Processo sia la parte reale che immaginaria dello spettrogramma.
    """
    def __init__(self, freq_bins, channels_mult=1, use_spectral_norm=True):
        super(SubDiscriminator, self).__init__()


        base_channels = 32
        channels = [
            base_channels * channels_mult,
            base_channels * 2 * channels_mult,
            base_channels * 4 * channels_mult,
            base_channels * 8 * channels_mult,
            base_channels * 8 * channels_mult
        ]

        # Scelgo se usare o meno gli spectral norm come layer
        def norm_layer(layer):
            return torch.nn.utils.spectral_norm(layer) if use_spectral_norm else layer

        # Encoder per parte reale
        self.real_conv1 = norm_layer(nn.Conv2d(1, channels[0], kernel_size=(7, 5), stride=(2, 2), padding=(3, 2)))
        self.real_conv2 = norm_layer(nn.Conv2d(channels[0], channels[1], kernel_size=(5, 3), stride=(2, 2), padding=(2, 1)))
        self.real_conv3 = norm_layer(nn.Conv2d(channels[1], channels[2], kernel_size=(3, 3), stride=(2, 1), padding=(1, 1)))
        self.real_conv4 = norm_layer(nn.Conv2d(channels[2], channels[3], kernel_size=(3, 3), stride=(2, 1), padding=(1, 1)))

        # Encoder per parte immaginaria
        self.imag_conv1 = norm_layer(nn.Conv2d(1, channels[0], kernel_size=(7, 5), stride=(2, 2), padding=(3, 2)))
        self.imag_conv2 = norm_layer(nn.Conv2d(channels[0], channels[1], kernel_size=(5, 3), stride=(2, 2), padding=(2, 1)))
        self.imag_conv3 = norm_layer(nn.Conv2d(channels[1], channels[2], kernel_size=(3, 3), stride=(2, 1), padding=(1, 1)))
        self.imag_conv4 = norm_layer(nn.Conv2d(channels[2], channels[3], kernel_size=(3, 3), stride=(2, 1), padding=(1, 1)))

        # Layer di fusione per combinare le feature reali e immaginarie
        self.fusion_conv = norm_layer(nn.Conv2d(channels[3] * 2, channels[4], kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)))

        # Layer finale per il punteggio di discriminazione
        self.final_conv = nn.Conv2d(channels[4], 1, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))


        # Batch normalization layers
        self.bn1 = nn.BatchNorm2d(channels[0])
        self.bn2 = nn.BatchNorm2d(channels[1])
        self.bn3 = nn.BatchNorm2d(channels[2])
        self.bn4 = nn.BatchNorm2d(channels[3])
        self.bn_fusion = nn.BatchNorm2d(channels[4])

        self.dropout = nn.Dropout2d(0.2)

    def forward(self, stft_complex):

        real = stft_complex.real  # [batch, freq_bins, time_frames]
        imag = stft_complex.imag  # [batch, freq_bins, time_frames]

        real = real.transpose(1, 2)  # [batch, time, freq]
        imag = imag.transpose(1, 2)  # [batch, time, freq]


        if len(real.shape) == 3:
            real = real.unsqueeze(1)  # [batch, 1, time, freq]
        if len(imag.shape) == 3:
            imag = imag.unsqueeze(1)  # [batch, 1, time, freq]

        # Processo parte reale
        r1 = F.leaky_relu(self.bn1(self.real_conv1(real)), 0.2)
        r2 = F.leaky_relu(self.bn2(self.real_conv2(r1)), 0.2)
        r3 = F.leaky_relu(self.bn3(self.real_conv3(r2)), 0.2)
        r4 = F.leaky_relu(self.bn4(self.real_conv4(r3)), 0.2)
        r4 = self.dropout(r4)

        # Processo parte immaginaria
        i1 = F.leaky_relu(self.bn1(self.imag_conv1(imag)), 0.2)
        i2 = F.leaky_relu(self.bn2(self.imag_conv2(i1)), 0.2)
        i3 = F.leaky_relu(self.bn3(self.imag_conv3(i2)), 0.2)
        i4 = F.leaky_relu(self.bn4(self.imag_conv4(i3)), 0.2)
        i4 = self.dropout(i4)

        # Concateno features reali e immaginarie
        combined = torch.cat([r4, i4], dim=1)

        # Fusione e output finale
        fused = F.leaky_relu(self.bn_fusion(self.fusion_conv(combined)), 0.2)
        output = self.final_conv(fused)

        return output, [r1, r2, r3, r4, i1, i2, i3, i4, fused]  # Ritorno features intermedie per feature matching loss


class MultiScaleDiscriminator(nn.Module):
    """
    Discriminatore Multi-Scala per Audio Super Resolution.
    Utilizza 3 discriminatori a scale diverse per catturare dettagli a varie risoluzioni temporali.
    """
    def __init__(self, freq_bins, num_scales=3, use_spectral_norm=True):
        super(MultiScaleDiscriminator, self).__init__()
        self.num_scales = num_scales
        self.freq_bins = freq_bins

        # Creo discriminatori per ogni scala
        self.discriminators = nn.ModuleList()
        for i in range(num_scales):
            # Ogni scala ha un moltiplicatore di canali diverso per catturare features diverse
            channels_mult = 2 ** i
            self.discriminators.append(
                SubDiscriminator(freq_bins, channels_mult, use_spectral_norm)
            )

        # Pooling layers per downsampling tra le scale
        self.downsample = nn.AvgPool2d(kernel_size=(2, 1), stride=(2, 1))

    def forward(self, stft_complex):
        """
        Forward pass attraverso tutti i discriminatori a scale multiple.

        Args:
            real: Parte reale dello spettrogramma [batch, time, freq]
            imag: Parte immaginaria dello spettrogramma [batch, time, freq]

        Returns:
            outputs: Lista di output da ogni discriminatore
            features: Lista di features intermedie per feature matching loss
        """
        outputs = []
        features = []

        current_input = stft_complex

        for i, disc in enumerate(self.discriminators):
            # Passo attraverso il discriminatore
            output, feat = disc(current_input)
            outputs.append(output)
            features.append(feat)

            # Downsample per la prossima scala (tranne l'ultima)
            if i < self.num_scales - 1:
                # Separo parti per downsampling
                real = current_input.real.transpose(1, 2).unsqueeze(1)  # [batch, 1, time, freq]
                imag = current_input.imag.transpose(1, 2).unsqueeze(1)

                # Applico downsampling
                real = self.downsample(real).squeeze(1).transpose(1, 2) # Torna a [batch, freq, time]
                imag = self.downsample(imag).squeeze(1).transpose(1, 2)

                # Ricostruisco il tensore complesso
                current_input = torch.complex(real, imag)

        return outputs, features


# GAN

In [ ]:


import os
import shutil
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm
import gc

def _move_obj_to_device(obj, device):
    if isinstance(obj, torch.Tensor):
        try:
            return obj.to(device)
        except Exception:
            return obj
    elif isinstance(obj, dict):
        for k, v in list(obj.items()):
            obj[k] = _move_obj_to_device(v, device)
        return obj
    elif isinstance(obj, (list, tuple)):
        seq = [_move_obj_to_device(x, device) for x in obj]
        return type(obj)(seq)
    else:
        return obj

def move_optimizer_state_to_device(opt, device):   
    for param_id, state in list(opt.state.items()):
        if isinstance(state, dict):
            for k, v in list(state.items()):
                state[k] = _move_obj_to_device(v, device)

# ===================== CONFIGURAZIONE =====================

if torch.cuda.is_available() and torch.cuda.device_count() >= 2:
    device = torch.device('cuda:0')   
    torch.cuda.set_device(device)
elif torch.cuda.is_available():
    device = torch.device('cuda')     
else:
    device = torch.device('cpu')

wave_dir = '/kaggle/temp/wave'
stft_dir='/kaggle/temp/stft'

batch_size = 2
accumulation_steps = 3
epochs = 30

lr_g = 2e-3
lr_d = 2e-3 * 0.3


alpha = 0.5   
beta = 0.9    
adv_weight = 0.7

lambda_feat = 1.0  
lambda_gp = 0.4   


d_steps = 2  
d_warmup_epochs = 3
d_steps_warmup = 3


# -------------------------------------------------------------------


use_discriminator = True
checkpoint_every = 5  # Salva checkpoint ogni N epoche

#torch.cuda.empty_cache()
if device.type == 'cuda':
    # Uso fino al 95% della VRAM disponibile (solo se CUDA presente)
    try:
        torch.cuda.set_per_process_memory_fraction(0.95)
    except Exception:
        pass
    torch.backends.cudnn.benchmark = True
    try:
        torch.backends.cuda.matmul.allow_tf32 = True
    except Exception:
        pass



class SpectralCoherenceLoss(nn.Module):
    def __init__(self):
        super(SpectralCoherenceLoss, self).__init__()

    def forward(self, real_pred, imag_pred, real_target, imag_target):
        assert real_pred.shape == imag_pred.shape, f"real_pred shape {real_pred.shape} != imag_pred shape {imag_pred.shape}"
        assert real_target.shape == imag_target.shape, f"real_target shape {real_target.shape} != imag_target shape {imag_target.shape}"
        assert real_pred.shape == real_target.shape, f"pred shape {real_pred.shape} != target shape {real_target.shape}"

        stft_pred = torch.complex(real_pred, imag_pred)
        stft_target = torch.complex(real_target, imag_target)

        pred_diff = stft_pred[:, :, 1:] - stft_pred[:, :, :-1]
        target_diff = stft_target[:, :, 1:] - stft_target[:, :, :-1]

        coherence_loss = F.mse_loss(pred_diff.abs(), target_diff.abs())
        return coherence_loss

class CombinedLoss(nn.Module):
    def __init__(self, alpha=1.0, beta=0.1):
        super(CombinedLoss, self).__init__()
        self.alpha = alpha
        self.beta = beta
        self.spectral_coherence_loss = SpectralCoherenceLoss()

    def forward(self, real_pred, imag_pred, real_target, imag_target):
        mse_loss = F.mse_loss(real_pred, real_target) + F.mse_loss(imag_pred, imag_target)
        coherence_loss = self.spectral_coherence_loss(real_pred, imag_pred, real_target, imag_target)
        total_loss = self.alpha * mse_loss + self.beta * coherence_loss
        return total_loss, mse_loss, coherence_loss

class DiscriminatorLoss(nn.Module):
    def __init__(self, lambda_feat=10.0, lambda_gp=10.0):
        super(DiscriminatorLoss, self).__init__()
        self.lambda_feat = lambda_feat
        self.lambda_gp = lambda_gp

    def adversarial_loss(self, disc_real, disc_fake, loss_type='wgan'):
        if loss_type == 'lsgan':
            real_target = 0.9
            loss_real = torch.mean((disc_real - real_target) ** 2)
            loss_fake = torch.mean(disc_fake ** 2)
        elif loss_type == 'wgan':
            loss_real = -torch.mean(disc_real)
            loss_fake = torch.mean(disc_fake)
        else:
            loss_real = F.binary_cross_entropy_with_logits(disc_real, torch.ones_like(disc_real))
            loss_fake = F.binary_cross_entropy_with_logits(disc_fake, torch.zeros_like(disc_fake))
        return loss_real + loss_fake

    def feature_matching_loss(self, feat_real, feat_fake):
        loss = 0
        for fr, ff in zip(feat_real, feat_fake):
            loss += torch.mean(torch.abs(fr.detach() - ff))
        return loss

    def gradient_penalty(self, discriminator, real_stft, fake_stft):
        batch_size = real_stft.size(0)
        alpha = torch.rand(batch_size, 1, 1).to(real_stft.device)
        interpolated = alpha * real_stft + (1 - alpha) * fake_stft
        interpolated.requires_grad_(True)
        disc_interpolated, _ = discriminator(interpolated)
        gradients = torch.autograd.grad(
            outputs=disc_interpolated[0].sum(),
            inputs=interpolated,
            create_graph=True,
            retain_graph=True
        )[0]
        gradient_norm = gradients.view(batch_size, -1).norm(2, dim=1)
        gp = ((gradient_norm - 1) ** 2).mean()
        return gp

    def forward(self, discriminator, real_stft, fake_stft, use_gp=False):
        disc_real_outputs, feat_real = discriminator(real_stft)
        disc_fake_outputs, feat_fake = discriminator(fake_stft.detach())

        adv_loss = 0
        for dr, df in zip(disc_real_outputs, disc_fake_outputs):
            adv_loss += self.adversarial_loss(dr, df, 'wgan')

        feat_loss = 0
        for fr, ff in zip(feat_real, feat_fake):
            feat_loss += self.feature_matching_loss(fr, ff)

        if use_gp:
            gp = self.gradient_penalty(discriminator, real_stft, fake_stft)
            total_loss = adv_loss + self.lambda_feat * feat_loss + self.lambda_gp * gp
            losses_dict = {'adv_loss': adv_loss.item(), 'feat_loss': feat_loss.item(), 'gp': gp.item()}
        else:
            total_loss = adv_loss + self.lambda_feat * feat_loss
            losses_dict = {'adv_loss': adv_loss.item(), 'feat_loss': feat_loss.item()}

        return total_loss, losses_dict

# ===================== FUNZIONI DI UTILITÀ =====================
def clear_gpu_memory():
    """Utility per liberare memoria GPU prima dell'addestramento"""
    if device.type == 'cuda':
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        gc.collect()
        total_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
        allocated = torch.cuda.memory_allocated() / 1e9
        free = total_memory - allocated
        print(f"Stato memoria GPU:")
        print(f"  Totale: {total_memory:.2f}GB")
        print(f"  Allocata: {allocated:.2f}GB")
        print(f"  Libera: {free:.2f}GB")
    else:
        gc.collect()
        print("Esecuzione su CPU: memoria CUDA non disponibile.")

# ===================== MAIN =====================
def main():
    clear_gpu_memory()
    print(f"Dispositivo: {device}")

    # Carico dataset
    from glob import glob
    audio_files = sorted(glob(os.path.join(wave_dir, '*.npy')))
    stft_files = sorted(glob(os.path.join(stft_dir, '*.npy')))

    if len(audio_files) == 0 or len(stft_files) == 0:
        raise ValueError(f"Nessun file trovato in {wave_dir} o {stft_dir}")

    print(f"Trovati {len(audio_files)} file audio, {len(stft_files)} file stft")

    
    dataset = AudioSTFTDataset(audio_files, stft_files)

    # Suddivido il dataset
    total = len(dataset)
    train_size = int(0.8 * total)
    val_size = total - train_size

    gen = torch.Generator().manual_seed(42)
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=gen)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=(device.type=='cuda'))
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, pin_memory=(device.type=='cuda'))

    print(f"Addestramento: {len(train_dataset)}, Validazione: {len(val_dataset)}")

    
    sample_batch = next(iter(train_loader))
    _, stft_in, stft_gt, _ = sample_batch

    # Stampo le shape
    print(f"\nForme di uscita del dataset:")
    print(f"  forma stft_in: {stft_in.shape}")
    print(f"  forma stft_gt: {stft_gt.shape}")

    B = stft_in.shape[0]
    # Controllo se è [B, F, T] o [B, T, F]
    if stft_in.shape[1] == 2049:
        freq_bins = stft_in.shape[1]
        time_frames = stft_in.shape[2]
        data_format = "BFT"
    else:
        time_frames = stft_in.shape[1]
        freq_bins = stft_in.shape[2]
        data_format = "BTF"

    print(f"Formato rilevato: {data_format}")
    print(f"Forma batch: B={B}, freq_bins={freq_bins}, time_frames={time_frames}")

    # Generatore
    G = ReteRecWithAttention(
        window_size=4096,
        freq_bins=freq_bins,
        hidden_size=128
    )

    print(f"Parametri generatore: {sum(p.numel() for p in G.parameters()):,}")

    
    D = None
    opt_d = None

    if use_discriminator:
        D = MultiScaleDiscriminator(
            freq_bins=freq_bins,
            num_scales=3,
            use_spectral_norm=True
        )
        print(f"Parametri discriminatore: {sum(p.numel() for p in D.parameters()):,}")
        opt_d = torch.optim.Adam(D.parameters(), lr=lr_d, betas=(0.5, 0.999))
    else:
        print("Addestramento senza discriminatore (solo loss L1)")

    
    opt_g = torch.optim.Adam(G.parameters(), lr=lr_g, betas=(0.5, 0.999))

    start_epoch = 0
    start_batch = 0
    accum_counter = 0
    scaler_g_state = None
    scaler_d_state = None

    
    checkpoint_path = '/kaggle/working/gan_checkpoint_final_run2_1587.pt'  
    history = {
        'g_total_loss': [],
        'g_mse_loss': [],
        'g_coherence_loss': [],
        'g_adv_loss': [],
        'd_total_loss': [],
        'd_adv_loss': [],
        'd_feat_loss': []
    }

   
    if os.path.exists(checkpoint_path):
        print(f"Caricamento checkpoint (solo pesi) da {checkpoint_path} ...")
        try:
            ckpt = torch.load(checkpoint_path, map_location=device)
        except Exception as e:
            print(f"Attenzione: errore map_location={device}, ricaduta su CPU: {e}")
            ckpt = torch.load(checkpoint_path, map_location='cpu')
    
        # Verifico NaN/Inf in uno state_dict (ritorna lista di tuple)
        def state_dict_has_invalids(state_dict):
            bad = []
            for name, tensor in state_dict.items():
                if isinstance(tensor, torch.Tensor):
                    if not torch.isfinite(tensor).all():
                        bad.append((name, float(torch.isnan(tensor).sum()), float(torch.isinf(tensor).sum())))
            return bad
    
        if 'G_state' in ckpt:
            bad_g = state_dict_has_invalids(ckpt['G_state'])
            if bad_g:
                print("ATTENZIONE: G_state contiene NaN/Inf in alcune voci. Elenco (nome, #NaN, #Inf):")
                for b in bad_g[:10]:
                    print(" ", b)
                raise RuntimeError("Checkpoint G_state corrotto: contiene NaN/Inf. Non carico i pesi.")
            try:
                G.load_state_dict(ckpt['G_state'])
                print("Generator: pesi caricati (solo G_state).")
            except Exception as e:
                print("Warning: errore caricamento G_state con strict=True:", e)
                G.load_state_dict(ckpt['G_state'], strict=False)
                print("Generator: caricamento con strict=False completato.")
        else:
            print("Nessun G_state nel checkpoint: il modello del generatore rimane inizializzato da zero.")
    
        
        if use_discriminator and D is not None:
            if 'D_state' in ckpt:
                bad_d = state_dict_has_invalids(ckpt['D_state'])
                if bad_d:
                    print("ATTENZIONE: D_state contiene NaN/Inf in alcune voci. Elenco (nome, #NaN, #Inf):")
                    for b in bad_d[:10]:
                        print(" ", b)
                    raise RuntimeError("Checkpoint D_state corrotto: contiene NaN/Inf. Non carico i pesi di D.")
                try:
                    D.load_state_dict(ckpt['D_state'])
                    print("Discriminator: pesi caricati (solo D_state).")
                except Exception as e:
                    print("Warning: errore caricamento D_state con strict=True:", e)
                    D.load_state_dict(ckpt['D_state'], strict=False)
                    print("Discriminator: caricamento con strict=False completato.")
            else:
                print("Nessun D_state nel checkpoint: il discriminatore rimane inizializzato da zero (se usato).")
    
        
        if 'opt_g' in ckpt:
            try:
                opt_g.load_state_dict(ckpt['opt_g'])
                move_optimizer_state_to_device(opt_g, device)
                print("opt_g stato ripristinato e spostato su device.")
            except Exception as e:
                print("Warning: impossibile caricare opt_g:", e)
    
        if use_discriminator and opt_d is not None and 'opt_d' in ckpt:
            try:
                opt_d.load_state_dict(ckpt['opt_d'])
                move_optimizer_state_to_device(opt_d, device)
                print("opt_d stato ripristinato e spostato su device.")
            except Exception as e:
                print("Warning: impossibile caricare opt_d:", e)
    
        # scaler
        if 'scaler_g' in ckpt and scaler_g is not None:
            try:
                scaler_g.load_state_dict(ckpt['scaler_g'])
                print("scaler_g ripristinato.")
            except Exception as e:
                print("Warning: impossibile caricare scaler_g:", e)
        if 'scaler_d' in ckpt and scaler_d is not None:
            try:
                scaler_d.load_state_dict(ckpt['scaler_d'])
                print("scaler_d ripristinato.")
            except Exception as e:
                print("Warning: impossibile caricare scaler_d:", e)
    
        start_epoch = ckpt.get('epoch', 0)
        start_batch = ckpt.get('batch_idx_in_epoch', 0)
        accum_counter = ckpt.get('accumulation_counter', 0)
    
        if start_epoch > 0:
            for g in opt_g.param_groups:
                old = g.get('lr', lr_g)
                g['lr'] = old * 0.1
            if opt_d is not None:
                for g in opt_d.param_groups:
                    old = g.get('lr', lr_d)
                    g['lr'] = old * 0.1
            print(f"Ripristino pesi da checkpoint epoca {start_epoch}. Ottimizzatori reinizializzati e LR ridotti (x0.1) per sicurezza.")
        else:
            print("Partenza da zero (start_epoch == 0). Ottimizzatori reinizializzati con LR di default.")
    
    else:
        print(f"Nessun checkpoint trovato in {checkpoint_path} — training partire da zero.")
        start_epoch = 0
        start_batch = 0
        accum_counter = 0

    import gc
    gc.collect()
    torch.cuda.empty_cache()


    # ================================================================================================

    G.to(device)
    if use_discriminator and D is not None:
        D.to(device)

    # Loss
    g_loss_fn = CombinedLoss(alpha=alpha, beta=beta)
    if use_discriminator:
        d_loss_fn = DiscriminatorLoss(lambda_feat=lambda_feat, lambda_gp=lambda_gp)

    print("\n=== Avvio addestramento ===")
    print(f"Batch size: {batch_size}, Accumulation steps: {accumulation_steps}")
    print(f"Batch efficace: {batch_size * accumulation_steps}")
    print(f"Uso discriminatore: {use_discriminator}")
    print(f"Loss generatore: CombinedLoss (alpha={g_loss_fn.alpha}, beta={g_loss_fn.beta})")
    if use_discriminator:
        print(f"Loss discriminatore: DiscriminatorLoss (lambda_feat={d_loss_fn.lambda_feat})")

    
    scaler_g = torch.cuda.amp.GradScaler() if device.type == 'cuda' else None
    scaler_d = torch.cuda.amp.GradScaler() if (use_discriminator and device.type == 'cuda') else None

    
    if device.type == 'cuda' and scaler_g is not None and 'scaler_g_state' in locals() and scaler_g_state is not None:
        try:
            scaler_g.load_state_dict(scaler_g_state)
            print("scaler_g ripristinato dal checkpoint.")
        except Exception as e:
            print(f"Warning: non è stato possibile caricare scaler_g: {e}")
    if device.type == 'cuda' and scaler_d is not None and 'scaler_d_state' in locals() and scaler_d_state is not None:
        try:
            scaler_d.load_state_dict(scaler_d_state)
            print("scaler_d ripristinato dal checkpoint.")
        except Exception as e:
            print(f"Warning: non è stato possibile caricare scaler_d: {e}")

    
    if 'history' not in locals():
        history = {
            'g_total_loss': [],
            'g_mse_loss': [],
            'g_coherence_loss': [],
            'g_adv_loss': [],
            'd_total_loss': [],
            'd_adv_loss': [],
            'd_feat_loss': []
        }

    
    training_data_format = data_format

    for epoch in range(start_epoch, epochs):
        G.train()
        if use_discriminator and D is not None:
            D.train()

        epoch_g_loss = 0
        epoch_g_mse = 0
        epoch_g_coherence = 0
        epoch_g_adv = 0
        epoch_d_loss = 0
        epoch_d_adv = 0
        epoch_d_feat = 0

        pbar = tqdm(train_loader, desc=f"Epoca {epoch+1}/{epochs}")

        for batch_idx, batch in enumerate(pbar):
            # Pulizia della cache periodica per evitare crash

            if epoch == start_epoch and batch_idx < start_batch:
                continue
            if batch_idx % 10 == 0 and device.type == 'cuda':
                torch.cuda.empty_cache()

            _, stft_in, stft_gt, _ = batch

            stft_in = stft_in.to(device)
            stft_gt = stft_gt.to(device)

            if training_data_format == "BFT":
                in_real = stft_in.real.permute(0, 2, 1).contiguous()
                in_imag = stft_in.imag.permute(0, 2, 1).contiguous()
                gt_real = stft_gt.real
                gt_imag = stft_gt.imag
            else:
                in_real = stft_in.real.contiguous()
                in_imag = stft_in.imag.contiguous()
                gt_real = stft_gt.real.contiguous()
                gt_imag = stft_gt.imag.contiguous()

            if use_discriminator and D is not None:
                with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                    with torch.no_grad():
                        fake_real, fake_imag = G(in_real, in_imag)

                        if training_data_format == "BFT":
                            fake_real_for_disc = fake_real.permute(0, 2, 1).contiguous()
                            fake_imag_for_disc = fake_imag.permute(0, 2, 1).contiguous()
                            real_stft = torch.complex(gt_real, gt_imag)
                        else:
                            fake_real_for_disc = fake_real.permute(0, 2, 1).contiguous()
                            fake_imag_for_disc = fake_imag.permute(0, 2, 1).contiguous()
                            real_stft = torch.complex(
                                gt_real.permute(0, 2, 1).contiguous(),
                                gt_imag.permute(0, 2, 1).contiguous()
                            )

                        fake_stft = torch.complex(fake_real_for_disc, fake_imag_for_disc)

                    _current_d_steps = d_steps if epoch >= d_warmup_epochs else d_steps_warmup

                    for _d_step in range(_current_d_steps):
                        if opt_d is not None:
                            opt_d.zero_grad()
                    
                        d_loss, d_losses_dict = d_loss_fn(D, real_stft, fake_stft, use_gp=True)
                        d_loss = d_loss / accumulation_steps
                    
                        if scaler_d is not None:
                            scaler_d.scale(d_loss).backward()
                        else:
                            d_loss.backward()
                    
                    
                    
                        torch.nn.utils.clip_grad_norm_(D.parameters(), max_norm=1.0)
                    

                        if (batch_idx + 1) % accumulation_steps == 0 and opt_d is not None:
                            if scaler_d is not None:
                                scaler_d.unscale_(opt_d)
                                torch.nn.utils.clip_grad_norm_(D.parameters(), max_norm=1.0)
                                scaler_d.step(opt_d)
                                scaler_d.update()
                            else:
                                torch.nn.utils.clip_grad_norm_(D.parameters(), max_norm=1.0)
                                opt_d.step()
                    
                        epoch_d_loss += d_loss.item() * accumulation_steps
                        epoch_d_adv += d_losses_dict['adv_loss']
                        epoch_d_feat += d_losses_dict['feat_loss']
                    
                                

            # ============ Addestramento Generatore ============
            # Freeze discriminator params so we don't update them while computing adv loss for G
            if use_discriminator and D is not None:
                for p in D.parameters():
                    p.requires_grad = False
                D.eval()  # evita aggiornamenti di running stats (BatchNorm) durante il forward di G
            
            with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                # Forward generator
                fake_real, fake_imag = G(in_real, in_imag)
            
                if batch_idx == 0 and epoch == 0:
                    print(f"\n=== Debug Generatore ===")
                    print(f"  Input a G forma: {in_real.shape}")
                    print(f"  Output da G forma: {fake_real.shape}")
                    print(f"  Ground truth forma: {gt_real.shape}")
                    print(f"  Formato dati: {training_data_format}")
            
                # Allinea shape se ci sono mismatch
                if training_data_format == "BFT":
                    gt_real_for_loss = gt_real.permute(0, 2, 1).contiguous()
                    gt_imag_for_loss = gt_imag.permute(0, 2, 1).contiguous()
                else:
                    gt_real_for_loss = gt_real
                    gt_imag_for_loss = gt_imag
            
                if fake_real.shape != gt_real_for_loss.shape:
                    print(f"ATTENZIONE: mismatch di shape - Output generatore: {fake_real.shape}, GT: {gt_real_for_loss.shape}")
                    min_t = min(fake_real.shape[1], gt_real_for_loss.shape[1])
                    min_f = min(fake_real.shape[2], gt_real_for_loss.shape[2])
                    fake_real = fake_real[:, :min_t, :min_f]
                    fake_imag = fake_imag[:, :min_t, :min_f]
                    gt_real_for_loss = gt_real_for_loss[:, :min_t, :min_f]
                    gt_imag_for_loss = gt_imag_for_loss[:, :min_t, :min_f]
            
                # Loss di ricostruzione + coerenza
                g_rec_total, g_mse, g_coherence = g_loss_fn(fake_real, fake_imag, gt_real_for_loss, gt_imag_for_loss)
            
                # Avversario: forward del discriminatore su fake (MA senza aggiornare i suoi pesi perché abbiamo requires_grad=False)
                g_adv_loss = torch.tensor(0.0, device=device)
                if use_discriminator and D is not None:
                    if training_data_format == "BFT":
                        fake_real_for_disc = fake_real.permute(0, 2, 1).contiguous()
                        fake_imag_for_disc = fake_imag.permute(0, 2, 1).contiguous()
                        real_stft = torch.complex(gt_real, gt_imag)
                    else:
                        fake_real_for_disc = fake_real.permute(0, 2, 1).contiguous()
                        fake_imag_for_disc = fake_imag.permute(0, 2, 1).contiguous()
                        # notare: qui usi permute come già fai nel codice esistente
                        real_stft = torch.complex(
                            gt_real.permute(0, 2, 1).contiguous(),
                            gt_imag.permute(0, 2, 1).contiguous()
                        )
            
                    fake_stft_g = torch.complex(fake_real_for_disc, fake_imag_for_disc)
                    # IMPORTANT: non usare torch.no_grad() qui: vogliamo il gradiente che torni a G
                    fake_outputs_g, _ = D(fake_stft_g)
            
                    for fake_out in fake_outputs_g:
                        g_adv_loss = g_adv_loss + (-torch.mean(fake_out)) 
            
                    # Combina ricostruzione + avversario (dividi per accumulation_steps come prima)
                    g_loss = (g_rec_total + adv_weight * g_adv_loss) / accumulation_steps
                else:
                    g_loss = g_rec_total / accumulation_steps
            
            # Backward (supporto mixed precision)
            if scaler_g is not None:
                scaler_g.scale(g_loss).backward()
            else:
                g_loss.backward()
            
            # Unfreeze discriminator params (ripristino allo stato per il training di D)
            if use_discriminator and D is not None:
                for p in D.parameters():
                    p.requires_grad = True
                D.train()
            
            # Optimizer step per G (gestito come prima con accumulation)
            if (batch_idx + 1) % accumulation_steps == 0:
                if scaler_g is not None:
                    scaler_g.unscale_(opt_g)
                    torch.nn.utils.clip_grad_norm_(G.parameters(), max_norm=1.0)
                    scaler_g.step(opt_g)
                    scaler_g.update()
                else:
                    torch.nn.utils.clip_grad_norm_(G.parameters(), max_norm=1.0)
                    opt_g.step()
                opt_g.zero_grad()
            
            # Aggiornamento statistiche per logging (mantieni come prima)
            epoch_g_loss += g_loss.item() * accumulation_steps
            epoch_g_mse += g_mse.item()
            epoch_g_coherence += g_coherence.item()
            if use_discriminator:
                epoch_g_adv += g_adv_loss.item()
            
            # libera
            del fake_real, fake_imag, fake_stft_g

            # Aggiornamento progress bar
            try:
                if use_discriminator:
                    pbar.set_postfix({
                        'Disc': f"{epoch_d_loss/(batch_idx+1):.4f}",
                        'Gen': f"{epoch_g_loss/(batch_idx+1):.4f}",
                        'MSE': f"{epoch_g_mse/(batch_idx+1):.4f}",
                        'Coh': f"{epoch_g_coherence/(batch_idx+1):.4f}",
                        'Mem': f"{(torch.cuda.memory_allocated()/1e9):.1f}GB" if device=='cuda' else "CPU"
                    })
                else:
                    pbar.set_postfix({
                        'Gen': f"{epoch_g_loss/(batch_idx+1):.4f}",
                        'MSE': f"{epoch_g_mse/(batch_idx+1):.4f}",
                        'Coh': f"{epoch_g_coherence/(batch_idx+1):.4f}",
                        'Mem': f"{(torch.cuda.memory_allocated()/1e9):.1f}GB" if device=='cuda' else "CPU"
                    })
            except Exception:
                pass


        n_batches = len(train_loader)
        avg_g_loss = epoch_g_loss / max(1, n_batches)
        avg_g_mse = epoch_g_mse / max(1, n_batches)
        avg_g_coherence = epoch_g_coherence / max(1, n_batches)

        print(f"\n=== Riepilogo Epoca {epoch+1} ===")
        print(f"Generatore - Totale: {avg_g_loss:.4f}, MSE: {avg_g_mse:.4f}, Coerenza: {avg_g_coherence:.4f}")

        if use_discriminator:
            avg_d_loss = epoch_d_loss / max(1, n_batches)
            avg_d_adv = epoch_d_adv / max(1, n_batches)
            avg_d_feat = epoch_d_feat / max(1, n_batches)
            avg_g_adv = epoch_g_adv / max(1, n_batches)

            print(f"Generatore - Avversario: {avg_g_adv:.4f}")
            print(f"Discriminatore - Totale: {avg_d_loss:.4f}, Adv: {avg_d_adv:.4f}, Feat: {avg_d_feat:.4f}")

            history['d_total_loss'].append(avg_d_loss)
            history['d_adv_loss'].append(avg_d_adv)
            history['d_feat_loss'].append(avg_d_feat)
            history['g_adv_loss'].append(avg_g_adv)

        history['g_total_loss'].append(avg_g_loss)
        history['g_mse_loss'].append(avg_g_mse)
        history['g_coherence_loss'].append(avg_g_coherence)

        if device.type == 'cuda':
            print(f"Memoria GPU: {torch.cuda.memory_allocated()/1e9:.2f}GB / {torch.cuda.get_device_properties(0).total_memory/1e9:.2f}GB")
        else:
            print("Esecuzione su CPU.")


        if epoch % checkpoint_every == 0:
            G.eval()
            val_loss = 0
            val_mse = 0
            val_coherence = 0
            val_batches = min(5, len(val_loader))

            with torch.no_grad():
                for i, batch in enumerate(val_loader):
                    if i >= val_batches:
                        break

                    _, stft_in, stft_gt, _ = batch
                    stft_in = stft_in.to(device)
                    stft_gt = stft_gt.to(device)

                    in_real = stft_in.real.permute(0, 2, 1)
                    in_imag = stft_in.imag.permute(0, 2, 1)
                    gt_real = stft_gt.real.permute(0, 2, 1)
                    gt_imag = stft_gt.imag.permute(0, 2, 1)

                    with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                        fake_real, fake_imag = G(in_real, in_imag)
                        v_total, v_mse, v_coh = g_loss_fn(fake_real, fake_imag, gt_real, gt_imag)

                        val_loss += v_total.item()
                        val_mse += v_mse.item()
                        val_coherence += v_coh.item()

                    del stft_in, stft_gt, in_real, in_imag, fake_real, fake_imag

            avg_val_loss = val_loss / max(1, val_batches)
            avg_val_mse = val_mse / max(1, val_batches)
            avg_val_coherence = val_coherence / max(1, val_batches)

            print(f"\nValidazione (su {val_batches} batch):")
            print(f"  Loss totale: {avg_val_loss:.4f}")
            print(f"  Loss MSE: {avg_val_mse:.4f}")
            print(f"  Loss Coerenza: {avg_val_coherence:.4f}")

        torch.cuda.empty_cache()
        gc.collect()

        if (epoch + 1) % checkpoint_every == 0:
            checkpoint_name = f'gan_checkpoint_epoch_run2_test_{epoch+1}.pt'
            checkpoint = {
                'epoch': epoch + 1,                      
                'batch_idx_in_epoch': batch_idx,        
                'accumulation_counter': accum_counter, 
                'G_state': G.state_dict(),
                'opt_g': opt_g.state_dict(),
                'history': history
            }
            if use_discriminator and D is not None:
                checkpoint['D_state'] = D.state_dict()
                checkpoint['opt_d'] = opt_d.state_dict()
            if scaler_g is not None:
                checkpoint['scaler_g'] = scaler_g.state_dict()
            if scaler_d is not None:
                checkpoint['scaler_d'] = scaler_d.state_dict()

            torch.save(checkpoint, checkpoint_name)
            print(f"Checkpoint salvato: {checkpoint_name}")


    final_name = f'gan_checkpoint_final_run2_test_{epochs}.pt'
    final_checkpoint = {
        'G_state': G.state_dict(),
        'opt_g': opt_g.state_dict(),
        'history': history
    }
    if use_discriminator and D is not None:
        final_checkpoint['D_state'] = D.state_dict()
        final_checkpoint['opt_d'] = opt_d.state_dict()
    if scaler_g is not None:
        final_checkpoint['scaler_g'] = scaler_g.state_dict()
    if scaler_d is not None:
        final_checkpoint['scaler_d'] = scaler_d.state_dict()

    torch.save(final_checkpoint, final_name)
    print(f"Salvato checkpoint finale in {final_name}")


    np.savez('training_history.npz', **history)

    print("\n=== Addestramento completato ===")
    print("Salvata storia dell'addestramento in training_history.npz")
    if device.type == 'cuda':
        print(f"Memoria GPU finale: {torch.cuda.memory_allocated()/1e9:.2f}GB utilizzata")
    else:
        print("Esecuzione su CPU.")

if __name__ == "__main__":
    main()
